# EXP007 — EXP005 FILE Stacker Submission Integration

EXP005 스태커를 v009 제출본에 넣되, 실제 제출에서 알 수 없는 합성 정답
`snr_db`는 **고정 0 dB**로 바꿉니다.

실행 순서:

1. EXP006 외부 feature에서 v009·Music×1.30·고정 0 dB 스태커를 비교
2. 외부 gate를 다시 통과할 때만 스태커 계수를 순수 NumPy NPZ로 변환
3. v009의 `FILE_FAKE_PROB`만 스태커 출력으로 교체
4. 나머지 네 출력 계산 코드는 유지
5. ZIP 구조·계수 일치·오프라인 계약을 검사
6. `submit_EXP007.zip` 생성

새 학습은 없으며 EXP005 체크포인트를 읽기만 합니다.

In [ ]:
# 0. Colab 의존성
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "joblib>=1.3", "scikit-learn>=1.3", "soundfile>=0.12.1",
    "demucs==4.0.1", "panns-inference==0.1.1", "safetensors>=0.6.2",
])
print("dependencies ready")

In [ ]:
# 1. 설정
from __future__ import annotations

import hashlib
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, roc_curve

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)

EXPERIMENT_ID = "EXP007"
CONFIG: dict[str, Any] = {
    "paths": {
        "output_root": "/content/drive/MyDrive/DeepV/experiments/EXP007",
        "submission_root": "/content/drive/MyDrive/DeepV/submissions",
        "work_root": "/content/exp007_build",
        "submit_zip_candidates": [
            "/content/drive/MyDrive/DeepV/submit.zip",
            "/content/drive/MyDrive/deepvoice/submit.zip",
            "/content/drive/MyDrive/submit.zip",
        ],
        "stacker_candidates": [
            "/content/drive/MyDrive/DeepV/experiments/EXP005/checkpoints/EXP005_file_stacker.joblib",
            "/content/drive/MyDrive/deepvoice/experiments/EXP005/checkpoints/EXP005_file_stacker.joblib",
        ],
        "exp006_features_candidates": [
            "/content/drive/MyDrive/DeepV/experiments/EXP006/features/EXP006_external_features.csv",
            "/content/drive/MyDrive/deepvoice/experiments/EXP006/features/EXP006_external_features.csv",
        ],
    },
    "gate": {
        "fixed_snr_db": 0.0,
        "minimum_eer_gain": 0.015,
        "auc_not_worse": True,
        "max_numpy_difference": 1e-7,
    },
    "run": {"build_submission": True},
}

OUTPUT_ROOT = Path(CONFIG["paths"]["output_root"])
SUBMISSION_ROOT = Path(CONFIG["paths"]["submission_root"])
WORK_ROOT = Path(CONFIG["paths"]["work_root"])
if not IN_COLAB:
    OUTPUT_ROOT = Path("./exp007_outputs").resolve()
    SUBMISSION_ROOT = OUTPUT_ROOT / "submissions"
    WORK_ROOT = OUTPUT_ROOT / "build"
for path in (OUTPUT_ROOT / "logs", OUTPUT_ROOT / "artifacts", SUBMISSION_ROOT, WORK_ROOT):
    path.mkdir(parents=True, exist_ok=True)


def first_file(values: list[str]) -> Path:
    for value in values:
        path = Path(value)
        if path.is_file():
            return path
    raise FileNotFoundError(f"required file missing: {values}")


SOURCE_ZIP = first_file(CONFIG["paths"]["submit_zip_candidates"])
STACKER_PATH = first_file(CONFIG["paths"]["stacker_candidates"])
EXP006_FEATURES_PATH = first_file(CONFIG["paths"]["exp006_features_candidates"])
FINAL_ZIP = SUBMISSION_ROOT / "submit_EXP007.zip"
print("v009:", SOURCE_ZIP)
print("stacker:", STACKER_PATH)
print("EXP006 features:", EXP006_FEATURES_PATH)
print("output:", FINAL_ZIP)

## 2. 실제 추론 가능한 고정 SNR 외부 gate

EXP006의 기존 결과는 합성 시 사용한 SNR을 특징으로 넣었습니다. 실제 제출에서는 이를
알 수 없으므로 `snr_db=0`으로 고정하여 다시 평가합니다. 이 gate가 실패하면 ZIP을 만들지 않습니다.

In [ ]:
PROBABILITY_FEATURES = [
    "voice_present", "music_present", "voice_df", "voice_fused",
    "music_fourier_raw", "music_fourier_stem", "v009_file",
]
RAW_FEATURES = ["vocal_rms", "music_stem_rms"]


def binary_metrics(target, probability) -> dict[str, float]:
    target = np.asarray(target, dtype=np.int64)
    probability = np.asarray(probability, dtype=np.float64)
    fpr, tpr, thresholds = roc_curve(
        target, probability, pos_label=1, drop_intermediate=False
    )
    fnr = 1.0 - tpr
    index = int(np.nanargmin(np.abs(fpr - fnr)))
    eer = float((fpr[index] + fnr[index]) / 2.0)
    threshold = float(thresholds[index])
    prediction = (probability >= threshold).astype(np.int64)
    return {
        "eer": eer, "eer_threshold": threshold,
        "roc_auc": float(roc_auc_score(target, probability)),
        "balanced_accuracy_at_eer": float(balanced_accuracy_score(target, prediction)),
    }


def feature_matrix(frame: pd.DataFrame, snr_db) -> np.ndarray:
    values = []
    for name in PROBABILITY_FEATURES:
        probability = np.clip(frame[name].to_numpy(float), 1e-5, 1.0 - 1e-5)
        values.extend([probability, np.log(probability / (1.0 - probability))])
    values.extend([
        frame["voice_present"].to_numpy(float) * frame["voice_fused"].to_numpy(float),
        frame["music_present"].to_numpy(float) * frame["music_fourier_raw"].to_numpy(float),
        frame["music_present"].to_numpy(float) * frame["music_fourier_stem"].to_numpy(float),
    ])
    values.extend(frame[name].to_numpy(float) for name in RAW_FEATURES)
    snr_values = np.asarray(snr_db, dtype=float)
    if snr_values.ndim == 0:
        snr_values = np.full(len(frame), float(snr_values), dtype=float)
    if snr_values.shape != (len(frame),):
        raise ValueError(f"invalid SNR feature shape: {snr_values.shape}")
    values.append(snr_values)
    return np.column_stack(values).astype(np.float64)


features = pd.read_csv(EXP006_FEATURES_PATH)
required = set(PROBABILITY_FEATURES + RAW_FEATURES + ["file_fake_label", "snr_db"])
missing = sorted(required - set(features.columns))
if missing:
    raise KeyError(f"EXP006 feature columns missing: {missing}")

stacker = joblib.load(STACKER_PATH)
y = features.file_fake_label.to_numpy(int)
baseline = features.v009_file.to_numpy(float)
music_scale_130 = np.clip(np.maximum(
    features.voice_present.to_numpy(float) * features.voice_fused.to_numpy(float),
    1.30 * features.music_present.to_numpy(float) * features.music_fourier_raw.to_numpy(float),
), 0.0, 1.0)
oracle = stacker.predict_proba(feature_matrix(features, features.snr_db.to_numpy(float)))[:, 1]
fixed_snr = float(CONFIG["gate"]["fixed_snr_db"])
candidate = stacker.predict_proba(feature_matrix(features, fixed_snr))[:, 1]
baseline_metrics = binary_metrics(y, baseline)
music_scale_130_metrics = binary_metrics(y, music_scale_130)
oracle_metrics = binary_metrics(y, oracle)
candidate_metrics = binary_metrics(y, candidate)
eer_gain = float(baseline_metrics["eer"] - candidate_metrics["eer"])
gate_pass = bool(
    np.isfinite(candidate).all()
    and eer_gain >= float(CONFIG["gate"]["minimum_eer_gain"])
    and candidate_metrics["roc_auc"] >= baseline_metrics["roc_auc"]
    and candidate_metrics["eer"] <= music_scale_130_metrics["eer"]
    and candidate_metrics["roc_auc"] >= music_scale_130_metrics["roc_auc"]
)
gate_result = {
    "rows": int(len(features)), "fixed_snr_db": fixed_snr,
    "v009": baseline_metrics,
    "exp057_music_scale_130_reference": music_scale_130_metrics,
    "exp005_oracle_snr_reference": oracle_metrics,
    "exp007_fixed_snr": candidate_metrics, "absolute_eer_gain": eer_gain,
    "official_score_proxy_gain_file_only": 0.45 * eer_gain,
    "gate_pass": gate_pass,
}
(OUTPUT_ROOT / "logs" / "EXP007_inference_safe_gate.json").write_text(
    json.dumps(gate_result, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(gate_result, ensure_ascii=False, indent=2))
if not gate_pass:
    raise RuntimeError("EXP007 fixed-SNR external gate failed; submission build blocked")

## 3. sklearn Pipeline을 순수 NumPy head로 변환

제출 코드에는 `joblib`과 `scikit-learn`을 추가하지 않습니다. StandardScaler와
LogisticRegression의 고정 파라미터만 작은 NPZ로 저장합니다.

In [ ]:
scale = stacker.named_steps["scale"]
model = stacker.named_steps["model"]
HEAD_DIR = WORK_ROOT / "model" / "file_stacker"
HEAD_DIR.mkdir(parents=True, exist_ok=True)
HEAD_PATH = HEAD_DIR / "weights.npz"
np.savez_compressed(
    HEAD_PATH,
    mean=np.asarray(scale.mean_, dtype=np.float64),
    scale=np.asarray(scale.scale_, dtype=np.float64),
    weight=np.asarray(model.coef_, dtype=np.float64).reshape(-1),
    bias=np.asarray(model.intercept_, dtype=np.float64).reshape(-1),
    fixed_snr_db=np.asarray([fixed_snr], dtype=np.float64),
    feature_count=np.asarray([20], dtype=np.int64),
)


def numpy_head_predict(matrix: np.ndarray, path: Path) -> np.ndarray:
    with np.load(path, allow_pickle=False) as payload:
        standardized = (matrix - payload["mean"]) / payload["scale"]
        logits = standardized @ payload["weight"] + float(payload["bias"][0])
    return 1.0 / (1.0 + np.exp(-np.clip(logits, -60.0, 60.0)))


numpy_candidate = numpy_head_predict(feature_matrix(features, fixed_snr), HEAD_PATH)
maximum_difference = float(np.max(np.abs(numpy_candidate - candidate)))
print("sklearn vs NumPy max difference:", maximum_difference)
if maximum_difference > float(CONFIG["gate"]["max_numpy_difference"]):
    raise RuntimeError("NumPy head conversion mismatch")

## 4. v009 script.py 패치

기존 음성·음악·presence 출력 계산은 유지합니다. 추가 계산은 분리된 music stem의
Fourier 점수와 두 stem RMS이며, 최종 FILE 출력만 NumPy 스태커로 바뀝니다.

In [ ]:
with zipfile.ZipFile(SOURCE_ZIP, "r") as archive:
    source_script = archive.read("script.py").decode("utf-8").replace("\r\n", "\n")


def replace_once(source: str, old: str, new: str, label: str) -> str:
    count = source.count(old)
    if count != 1:
        raise RuntimeError(f"{label}: expected one patch anchor, found {count}")
    return source.replace(old, new, 1)


old_combiner = '''def combine_file_fake_score(voice_fake, music_fake, voice_present, music_present):
    voice_score = voice_present * voice_fake
    music_score = music_present * music_fake
    return max(voice_score, music_score)
'''
new_combiner = old_combiner + '''

def load_file_stacker_head():
    path = MODEL_DIR / "file_stacker" / "weights.npz"
    if not path.is_file():
        raise FileNotFoundError(f"FILE stacker weights missing: {path}")
    with np.load(path, allow_pickle=False) as payload:
        head = {name: payload[name].copy() for name in payload.files}
    if int(head["feature_count"][0]) != 20:
        raise ValueError("Unexpected FILE stacker feature contract")
    return head


def _stacker_probability_features(values):
    output = []
    for value in values:
        probability = float(np.clip(value, 1e-5, 1.0 - 1e-5))
        output.extend([probability, float(np.log(probability / (1.0 - probability)))])
    return output


def predict_file_stacker(
    head, voice_present, music_present, voice_df, voice_fused,
    music_fourier_raw, music_fourier_stem, v009_file,
    vocal_rms, music_stem_rms,
):
    features = _stacker_probability_features([
        voice_present, music_present, voice_df, voice_fused,
        music_fourier_raw, music_fourier_stem, v009_file,
    ])
    features.extend([
        voice_present * voice_fused,
        music_present * music_fourier_raw,
        music_present * music_fourier_stem,
        vocal_rms, music_stem_rms, float(head["fixed_snr_db"][0]),
    ])
    vector = np.asarray(features, dtype=np.float64)
    if vector.size != int(head["feature_count"][0]) or not np.isfinite(vector).all():
        raise ValueError("Invalid FILE stacker feature vector")
    standardized = (vector - head["mean"]) / head["scale"]
    logit = float(np.dot(standardized, head["weight"]) + head["bias"][0])
    return float(1.0 / (1.0 + np.exp(-np.clip(logit, -60.0, 60.0))))
'''
patched = replace_once(source_script, old_combiner, new_combiner, "combiner")

old_load = '''    fourier_fakeprint_head = load_fourier_fakeprint_head(device)

    for index, audio_path in enumerate(tqdm(audio_files, desc="Components")):
'''
new_load = '''    fourier_fakeprint_head = load_fourier_fakeprint_head(device)
    file_stacker_head = load_file_stacker_head()

    for index, audio_path in enumerate(tqdm(audio_files, desc="Components")):
'''
patched = replace_once(patched, old_load, new_load, "head loading")

old_music = '''        if music_present - voice_present >= MUSIC_DOMINANCE_MARGIN:
            music_fake = predict_fourier_fakeprint(
                original_audio, fourier_fakeprint_head, device
            )
        else:
            music_fake = predict_fake(
                df_arena_model, fake_label_index, music_audio, device
            )
'''
new_music = '''        music_fourier_raw = predict_fourier_fakeprint(
            original_audio, fourier_fakeprint_head, device
        )
        music_fourier_stem = predict_fourier_fakeprint(
            music_audio, fourier_fakeprint_head, device
        )
        if music_present - voice_present >= MUSIC_DOMINANCE_MARGIN:
            music_fake = music_fourier_raw
        else:
            music_fake = predict_fake(
                df_arena_model, fake_label_index, music_audio, device
            )
'''
patched = replace_once(patched, old_music, new_music, "music features")

old_file = '''        file_fake = combine_file_fake_score(
            fused_voice_fake, music_fake, voice_present, music_present
        )
'''
new_file = '''        v009_file_fake = combine_file_fake_score(
            fused_voice_fake, music_fake, voice_present, music_present
        )
        file_fake = predict_file_stacker(
            file_stacker_head,
            voice_present, music_present, voice_fake, fused_voice_fake,
            music_fourier_raw, music_fourier_stem, v009_file_fake,
            calculate_rms(voice_audio), calculate_rms(music_audio),
        )
'''
patched = replace_once(patched, old_file, new_file, "FILE output")

compile(patched, "script.py", "exec")
PATCHED_SCRIPT = WORK_ROOT / "script.py"
PATCHED_SCRIPT.write_text(patched, encoding="utf-8")
print("patched script:", PATCHED_SCRIPT)

## 5. 제출 ZIP 생성과 정적 계약 검사

대형 가중치는 다시 압축하지 않도록 v009 ZIP을 복사한 뒤 `script.py`만 교체하고
작은 NumPy head를 추가합니다.

In [ ]:
def sha256(path: Path, chunk_size: int = 16 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def build_zip() -> None:
    local_zip = WORK_ROOT / "submit_EXP007.zip"
    shutil.copy2(SOURCE_ZIP, local_zip)
    subprocess.check_call(["zip", "-d", str(local_zip), "script.py"])
    subprocess.check_call(
        ["zip", "-9", str(local_zip), "script.py", "model/file_stacker/weights.npz"],
        cwd=WORK_ROOT,
    )
    with zipfile.ZipFile(local_zip, "r") as archive:
        names = archive.namelist()
        bad = archive.testzip()
        if bad is not None:
            raise RuntimeError(f"corrupt ZIP member: {bad}")
        if names.count("script.py") != 1 or names.count("model/file_stacker/weights.npz") != 1:
            raise RuntimeError("submission ZIP entry contract failed")
        archived_script = archive.read("script.py").decode("utf-8")
        compile(archived_script, "archived_script.py", "exec")
        if "predict_file_stacker" not in archived_script:
            raise RuntimeError("patched FILE stacker code missing")
        if 'row["VOICE_FAKE_PROB"] = round(fused_voice_fake, 10)' not in archived_script:
            raise RuntimeError("VOICE output contract changed")
        if 'row["MUSIC_FAKE_PROB"] = round(music_fake, 10)' not in archived_script:
            raise RuntimeError("MUSIC output contract changed")
        if 'row["VOICE_PRESENT_PROB"] = round(voice_present, 10)' not in archived_script:
            raise RuntimeError("VOICE presence contract changed")
        if 'row["MUSIC_PRESENT_PROB"] = round(music_present, 10)' not in archived_script:
            raise RuntimeError("MUSIC presence contract changed")
    shutil.copy2(local_zip, FINAL_ZIP)


if CONFIG["run"]["build_submission"]:
    build_zip()

summary = {
    "experiment_id": EXPERIMENT_ID,
    "source_submission": str(SOURCE_ZIP),
    "output_submission": str(FINAL_ZIP),
    "output_exists": FINAL_ZIP.is_file(),
    "output_bytes": int(FINAL_ZIP.stat().st_size) if FINAL_ZIP.is_file() else 0,
    "output_sha256": sha256(FINAL_ZIP) if FINAL_ZIP.is_file() else None,
    "training": False,
    "file_output_changed": True,
    "other_four_outputs_code_preserved": True,
    "sklearn_in_submission": False,
    "fixed_snr_db": fixed_snr,
    "external_gate": gate_result,
    "numpy_conversion_max_difference": maximum_difference,
    "submit_zip_modified_in_place": False,
    "ready_for_dacon_smoke": bool(gate_pass and FINAL_ZIP.is_file()),
}
(OUTPUT_ROOT / "logs" / "EXP007_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 완료 후 공유할 파일

```text
MyDrive/DeepV/experiments/EXP007/logs/EXP007_inference_safe_gate.json
MyDrive/DeepV/experiments/EXP007/logs/EXP007_summary.json
```

`ready_for_dacon_smoke=true`를 확인한 뒤 생성된 파일은 다음 위치에 있습니다.

```text
MyDrive/DeepV/submissions/submit_EXP007.zip
```

아직 바로 최종 제출로 확정하지 말고, 먼저 DACON 코드 검증/소량 smoke 제출에 사용합니다.